In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import TimeoutException, NoSuchElementException

import time
import pandas as pd
from pathlib import Path

options = Options()
options.add_argument("--start-maximized")
# 위치 제공 차단
options.add_experimental_option("prefs", {
  "profile.default_content_setting_values.geolocation": 2
})




In [ ]:
URL = "https://www.banapresso.com/"

driver = webdriver.Chrome(options=options)

driver.get(URL)

wait = WebDriverWait(driver, 5)

# 매장 찾기 버튼
button_1 = wait.until(
  EC.element_to_be_clickable(
    (By.XPATH, "/html/body/div/div/div/header/div/ul/li[2]/a")
  )
)


button_1.click()


try: 
  # 위치 권한 버튼
  button_1 = wait.until(
    EC.element_to_be_clickable(
      (By.XPATH, "/html/body/div/div[2]/div/div[2]/button")
    )
  )
  button_1.click()
except TimeoutException as te:
  # 없으면 오히려 좋아
  pass




# 로드 되길 기다리기
store_list = wait.until(
  EC.visibility_of_element_located(
    (By.CSS_SELECTOR, "div.drblfU.store_shop_list")
  )
)


# 드래그하기


# # # # # # # #  액션 체인은 안먹혀서 js로 사용  # # # # # # # # 
# actions = ActionChains(driver)
# actions \
#   .move_to_element(store_list) \
#   .scroll_by_amount(0, 2000) \
#   .perform
  # .scroll_by_amount(0, 2000).perform()

# # # # # # # #  js로 사용  # # # # # # # # 
# 위치 확인용 스크립트
for _ in range(30):
  driver.execute_script(
    # arguments[0] 은 셀레니움에서 넘겨준 첫번째 값을 의미
    # store_list는 스크롤 할 대상
    # scrollTop는 현재 스크롤 내려간 양
    # scrollHeight는 스크롤 가능한 전체 높이
    "arguments[0].scrollTop += arguments[0].scrollHeight",
    store_list # 찾은 ul 품은 div 요소
  )
  time.sleep(0.3)

# time.sleep(5)

# print(store_list.find_elements(By.TAG_NAME, "li"))

store_info_list = store_list.find_elements(By.TAG_NAME, "li")

# 이름 | 주소 | 오픈 시 | 오픈 분 | 종료 시 | 종료 분 | 주차 가능 여부
store_data = []

print("작업 시작")
store_count = len(store_info_list)
print(f"총 매장 개수: {store_count}")
idx = 1
for store in store_info_list:
  try: 
    print()
    print(f"\n --- ({idx}/{store_count}) 번째 작업 시작 --- \n")
    idx += 1
    print(store.text)

    # 매장 명
    store_name = store.find_element(By.CSS_SELECTOR, "p.name")
    print(store_name.text)

    # 위치
    store_addr = store.find_element(By.CSS_SELECTOR, "p.address")
    print(store_addr.text)

    # 영업 시간
    store_open = store.find_element(By.CSS_SELECTOR, "div.time")
    print(store_open.text)

    store_open = list(map(lambda s: s.split(":"),store_open.text.replace("OPEN\n", "").split("~")))

    # 주차 가능
    parkable = False
    try:
      store.find_element(By.CSS_SELECTOR, "div.parking")
      parkable = True
    except NoSuchElementException as e:
      pass

    store_data.append({
      "name": store_name.text,
      "addr": store_addr.text,
      "open_o_clock": int(store_open[0][0]),
      "open_minute": int(store_open[0][1]),
      "close_o_clock": int(store_open[1][0]),
      "close_minute": int(store_open[1][1]),
      "parkable": parkable
    })
    
  except Exception as e:
    print(e)
    if "Intersection Observer Trigger" in store.text:
      print("작업 완료")
      break
    print(store.text)


  print(f"\n --- ({idx}/{store_count}) 번째 작업 끝 --- \n")
  

  
df = pd.DataFrame(store_data, columns=store_data[0].keys())

with open((Path() / "bana_store_info.csv"), "w", encoding="utf-8-sig", newline="") as f:
  df.to_csv(f, index=False, encoding="utf-8")
# driver.quit()

작업 시작
총 매장 개수: 229


 --- (1/229) 번째 작업 시작 --- 

가락몰점
서울특별시 송파구 양재대로 932, 업무동 1층 로비
OPEN
07:30~23:30
가락몰점
서울특별시 송파구 양재대로 932, 업무동 1층 로비
OPEN
07:30~23:30

 --- (2/229) 번째 작업 끝 --- 



 --- (2/229) 번째 작업 시작 --- 

가산디지털단지역점
서울시 금천구 가산동 60-3
OPEN
07:00~19:00
가산디지털단지역점
서울시 금천구 가산동 60-3
OPEN
07:00~19:00

 --- (3/229) 번째 작업 끝 --- 



 --- (3/229) 번째 작업 시작 --- 

가산안양천점
서울 금천구 가산 디지털2로 127-143, 101호
OPEN
07:00~20:00
가산안양천점
서울 금천구 가산 디지털2로 127-143, 101호
OPEN
07:00~20:00

 --- (4/229) 번째 작업 끝 --- 



 --- (4/229) 번째 작업 시작 --- 

가산어반워크점
서울시 금천구 가산디지털2로 135, 1동 142호
CLOSE
07:00~17:30
가산어반워크점
서울시 금천구 가산디지털2로 135, 1동 142호
Message: no such element: Unable to locate element: {"method":"css selector","selector":"div.time.open"}
  (Session info: chrome=149.0.7827.196); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff650503fa5+14925]
	chromedriver!GetHandleVerifie

In [43]:
print((Path() / "pri.csv").cwd())

c:\2026-05-19_KDT_lang_chain\workspace\personal\assignment\07_bana_selenium
